# TRIBE v2 Surface Decoder UI

This UI-only Colab notebook launches Gradio. Setup, dependency install, TRIBE inference, cache reuse, surface decoding, and visualization are all triggered from the interface button.


In [ ]:
# -*- coding: utf-8 -*-
import base64
import json
import os
import pickle
import queue
import shutil
import socket
import subprocess
import sys
import threading
import time
import traceback
import zipfile
from pathlib import Path


def parse_version_prefix(raw: str) -> tuple[int, int]:
    """Return major/minor from a package version string."""

    parts = []
    for token in raw.replace("-", ".").split("."):
        if token.isdigit():
            parts.append(int(token))
        else:
            digits = "".join(char for char in token if char.isdigit())
            if digits:
                parts.append(int(digits))
        if len(parts) >= 2:
            break
    while len(parts) < 2:
        parts.append(0)
    return parts[0], parts[1]


def ensure_numpy_compatibility() -> None:
    """Pin NumPy before Gradio/TRIBE dependencies import it in Colab."""

    from importlib.metadata import PackageNotFoundError, version

    try:
        numpy_version = version("numpy")
    except PackageNotFoundError:
        numpy_version = ""

    needs_pin = not numpy_version or parse_version_prefix(numpy_version) >= (2, 1)
    if not needs_pin:
        print(f"NumPy compatible with TRIBE plotting: {numpy_version}")
        return

    numpy_already_loaded = "numpy" in sys.modules
    print(f"Installing NumPy <2.1 for TRIBE plotting compatibility. Current: {numpy_version or 'not installed'}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2.1"],
        check=True,
    )
    if numpy_already_loaded:
        raise RuntimeError(
            "NumPy was already imported in this kernel before it was pinned. "
            "Restart the Colab runtime once, then rerun this UI cell."
        )


ensure_numpy_compatibility()

print("Importing Gradio...", flush=True)
try:
    import gradio as gr
except ImportError:
    print("Gradio is not installed. Installing gradio>=4.44,<6...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "gradio>=4.44,<6", "numpy<2.1"], check=True)
    import gradio as gr
print(f"Gradio imported: {getattr(gr, '__version__', 'unknown')}", flush=True)

DEBUG_BUILD_ID = "hybrid-transcription-20260615-01"
PROJECT_DIR = Path("/content/neuromedia")
DEFAULT_ROOT_DIR = "/content/drive/MyDrive/neuromedia"
DEFAULT_INPUT_PATH = "/content/drive/MyDrive/neuromedia/input"
# --- Project source: cloned from GitHub at runtime (replaces embedded base64) ---
# Set REPO_URL to your public repo, e.g. "https://github.com/youruser/neuromedia.git"
REPO_URL = "https://github.com/CHex0K/neuromedia.git"
REPO_BRANCH = "main"
SUPPORTED_EXTENSIONS = {".txt", ".wav", ".mp3", ".flac", ".ogg", ".mp4", ".avi", ".mkv", ".mov", ".webm"}
BATCH_VIDEO_EXTENSIONS = {".mp4", ".avi"}


def find_open_port(start: int = 7860, attempts: int = 50) -> int:
    """Find a local TCP port for the Gradio server."""

    for port in range(start, start + attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("127.0.0.1", port))
            except OSError:
                continue
            return port
    raise RuntimeError(f"No free TCP port found in range {start}-{start + attempts - 1}")


def save_gradio_urls(url_text: str) -> None:
    """Persist Gradio URLs so VS Code/Colab output glitches do not hide them."""

    local_path = PROJECT_DIR / "gradio_url.txt"
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_text(url_text, encoding="utf-8")
        print(f"Saved Gradio URL to: {local_path}", flush=True)
    except Exception as exc:
        print(f"Could not save Gradio URL to {local_path}: {exc}", flush=True)

    drive_mount = Path("/content/drive")
    if not is_active_mount(drive_mount):
        print("Google Drive is not mounted; skipped Drive URL log.", flush=True)
        return

    drive_path = Path(DEFAULT_ROOT_DIR) / "logs" / "gradio_url.txt"
    try:
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        drive_path.write_text(url_text, encoding="utf-8")
        print(f"Saved Gradio URL to: {drive_path}", flush=True)
    except Exception as exc:
        print(f"Could not save Gradio URL to {drive_path}: {exc}", flush=True)


def write_project_files() -> None:
    """Clone the project into the Colab runtime from GitHub.

    Replaces the old embedded-base64 mechanism: edit the .py files, push to
    GitHub, and rerunning this cell pulls the latest code. No build step.
    """

    if REPO_URL.split("/")[-2] == "CHANGE_ME":
        raise RuntimeError("Set REPO_URL to your GitHub repo before running.")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    print(f"Cloning {REPO_URL}@{REPO_BRANCH} ...", flush=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
    (PROJECT_DIR / "build_id.txt").write_text(DEBUG_BUILD_ID, encoding="utf-8")
    print(f"Project cloned into {PROJECT_DIR}", flush=True)


def output_dir_for_input(input_path: Path, input_dir: Path, tribe_output_root: Path) -> Path:
    """Return stable TRIBE output directory for an input file."""

    try:
        relative = input_path.relative_to(input_dir).with_suffix("")
        return tribe_output_root.joinpath(*relative.parts)
    except ValueError:
        return tribe_output_root / input_path.stem


def format_gib_from_kib(value_kib: int | None) -> str:
    """Format a KiB value from /proc/meminfo as GiB."""

    if value_kib is None:
        return "n/a"
    return f"{value_kib / 1024 / 1024:.1f} GiB"


def read_meminfo() -> dict[str, int]:
    """Read selected /proc/meminfo values in KiB."""

    out: dict[str, int] = {}
    try:
        for line in Path("/proc/meminfo").read_text(encoding="utf-8").splitlines():
            key, raw_value = line.split(":", 1)
            value = raw_value.strip().split()[0]
            if value.isdigit():
                out[key] = int(value)
    except Exception:
        pass
    return out


def short_command_output(cmd: list[str], timeout: float = 5.0) -> str:
    """Run a small diagnostic command and return compact stdout/stderr."""

    try:
        completed = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=False,
            timeout=timeout,
        )
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}"
    text = (completed.stdout or completed.stderr or "").strip()
    return text or f"exit={completed.returncode}; no output"


def append_resource_snapshot(log_lines: list[str], label: str, pid: int | None = None) -> None:
    """Append RAM/GPU/process diagnostics to the UI log."""

    meminfo = read_meminfo()
    total = meminfo.get("MemTotal")
    available = meminfo.get("MemAvailable")
    used = total - available if total is not None and available is not None else None
    swap_total = meminfo.get("SwapTotal")
    swap_free = meminfo.get("SwapFree")
    swap_used = swap_total - swap_free if swap_total is not None and swap_free is not None else None
    log_lines.append(f"[diag] {label}")
    log_lines.append(
        "[diag] RAM used/total/available: "
        f"{format_gib_from_kib(used)} / {format_gib_from_kib(total)} / {format_gib_from_kib(available)}"
    )
    log_lines.append(
        "[diag] swap used/total/free: "
        f"{format_gib_from_kib(swap_used)} / {format_gib_from_kib(swap_total)} / {format_gib_from_kib(swap_free)}"
    )
    if pid is not None:
        ps_output = short_command_output([
            "ps",
            "-p",
            str(pid),
            "-o",
            "pid,ppid,stat,etime,%cpu,%mem,rss,vsz,cmd",
            "--no-headers",
        ])
        log_lines.append(f"[diag] child ps: {ps_output}")
    gpu_output = short_command_output([
        "nvidia-smi",
        "--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu",
        "--format=csv,noheader,nounits",
    ])
    log_lines.append(f"[diag] GPU name,mem_used,mem_total,util,temp: {gpu_output}")
    gpu_processes = short_command_output([
        "nvidia-smi",
        "--query-compute-apps=pid,process_name,used_memory",
        "--format=csv,noheader,nounits",
    ])
    log_lines.append(f"[diag] GPU processes pid,name,mem: {gpu_processes}")


TRIBE_WORKER_RESULT_PREFIX = "@@TRIBE_WORKER_RESULT@@ "
TRIBE_WORKER_MAX_RESTARTS = 3


class TribeWorkerCrashed(RuntimeError):
    """Raised when the persistent TRIBE worker process dies unexpectedly."""


class TribeWorkerSession:
    """Drive a long-lived tribe_worker.py so the TRIBE model loads once per run.

    Log lines from the worker stream into ``log_lines`` exactly like
    run_command_stream; only sentinel-prefixed lines are parsed as protocol
    results. ``start`` and ``run_job`` are generators that yield so the caller
    can refresh the Gradio UI while the worker loads / infers.
    """

    def __init__(self, cmd, log_lines, env=None, monitor_interval=15.0):
        self.cmd = cmd
        self.log_lines = log_lines
        self.env = env
        self.monitor_interval = monitor_interval
        self.process = None
        self.queue = None
        self.reader = None
        self.last_result = None

    def _drain_nowait(self):
        emitted = False
        while True:
            try:
                line = self.queue.get_nowait()
            except queue.Empty:
                break
            if line.startswith(TRIBE_WORKER_RESULT_PREFIX):
                payload = line[len(TRIBE_WORKER_RESULT_PREFIX):]
                try:
                    self.last_result = json.loads(payload)
                except Exception as exc:
                    self.log_lines.append(f"[worker] unparseable result line: {exc}: {payload}")
            else:
                self.log_lines.append(line)
                emitted = True
        return emitted

    def start(self):
        self.log_lines.append("$ " + " ".join(self.cmd))
        self.process = subprocess.Popen(
            self.cmd,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=self.env,
            bufsize=1,
        )
        self.queue = queue.Queue()

        def read_output():
            assert self.process.stdout is not None
            for raw_line in self.process.stdout:
                self.queue.put(raw_line.rstrip())

        self.reader = threading.Thread(target=read_output, daemon=True)
        self.reader.start()
        self.last_result = None
        while True:
            self._drain_nowait()
            if self.last_result is not None:
                status = self.last_result.get("status")
                if status == "ready":
                    self.log_lines.append("[worker] TRIBE model loaded and ready.")
                    self.last_result = None
                    return
                if status == "fatal":
                    raise TribeWorkerCrashed(
                        f"worker failed to load model: {self.last_result.get('error')}"
                    )
            if self.process.poll() is not None:
                self._drain_nowait()
                raise TribeWorkerCrashed(
                    f"worker exited during startup (code {self.process.returncode})"
                )
            yield
            time.sleep(0.2)

    def run_job(self, job):
        if self.process is None or self.process.poll() is not None:
            raise TribeWorkerCrashed("worker is not running")
        self.last_result = None
        try:
            self.process.stdin.write(json.dumps(job, ensure_ascii=False) + "\n")
            self.process.stdin.flush()
        except Exception as exc:
            raise TribeWorkerCrashed(f"could not send job to worker: {exc}")
        last_monitor = time.monotonic()
        while True:
            emitted = self._drain_nowait()
            if emitted:
                yield
            if self.last_result is not None:
                return
            if self.process.poll() is not None:
                self._drain_nowait()
                if self.last_result is None:
                    raise TribeWorkerCrashed(
                        f"worker crashed during job (code {self.process.returncode})"
                    )
                return
            now = time.monotonic()
            if now - last_monitor >= self.monitor_interval:
                append_resource_snapshot(
                    self.log_lines, f"monitor tribe worker pid={self.process.pid}", pid=self.process.pid
                )
                last_monitor = now
                yield
            if not emitted:
                time.sleep(0.3)

    def shutdown(self):
        try:
            if self.process is not None and self.process.poll() is None:
                try:
                    self.process.stdin.write(json.dumps({"command": "shutdown"}) + "\n")
                    self.process.stdin.flush()
                except Exception:
                    pass
                try:
                    self.process.wait(timeout=10)
                except Exception:
                    self.process.kill()
        except Exception as exc:
            self.log_lines.append(f"[worker] shutdown error: {exc}")


def run_command_stream_with_retries(cmd, log_lines, env=None, attempts=3, base_sleep=8.0):
    """Run a command with retry+backoff. Useful for flaky network installs
    (e.g. `pip install` cloning a git dependency from github.com that times out)."""

    for attempt in range(1, attempts + 1):
        try:
            for _ in run_command_stream(cmd, log_lines, env=env):
                yield
            return
        except subprocess.CalledProcessError as exc:
            if attempt >= attempts:
                raise
            sleep_s = base_sleep * attempt
            log_lines.append(
                f"[retry] command failed (exit {exc.returncode}); "
                f"attempt {attempt}/{attempts} failed, retrying in {sleep_s:.0f}s..."
            )
            yield
            time.sleep(sleep_s)


def run_command_stream(
    cmd: list[str],
    log_lines: list[str],
    env: dict[str, str] | None = None,
    monitor_interval: float = 15.0,
):
    """Run a command, stream output, and log resources while it is quiet."""

    log_lines.append("$ " + " ".join(cmd))
    append_resource_snapshot(log_lines, "before command")
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    output_queue: queue.Queue[str] = queue.Queue()

    def read_output() -> None:
        assert process.stdout is not None
        for raw_line in process.stdout:
            output_queue.put(raw_line.rstrip())

    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    append_resource_snapshot(log_lines, f"started child pid={process.pid}", pid=process.pid)
    last_monitor = time.monotonic()

    while True:
        emitted = False
        while True:
            try:
                log_lines.append(output_queue.get_nowait())
                emitted = True
            except queue.Empty:
                break
        if emitted:
            yield

        return_code = process.poll()
        now = time.monotonic()
        if now - last_monitor >= monitor_interval:
            append_resource_snapshot(log_lines, f"monitor child pid={process.pid}", pid=process.pid)
            last_monitor = now
            yield

        if return_code is not None:
            reader.join(timeout=1.0)
            while True:
                try:
                    log_lines.append(output_queue.get_nowait())
                except queue.Empty:
                    break
            if return_code != 0:
                if return_code < 0:
                    log_lines.append(f"[diag] command terminated by signal {-return_code}")
                append_resource_snapshot(log_lines, f"after failed command exit={return_code}", pid=process.pid)
                yield
                raise subprocess.CalledProcessError(return_code, cmd)
            append_resource_snapshot(log_lines, "after successful command")
            yield
            return

        if not emitted:
            time.sleep(0.5)


def is_active_mount(path: Path) -> bool:
    """Return True when path is an active OS mount point."""

    try:
        if os.path.ismount(path):
            return True
    except Exception:
        pass
    try:
        mounts = Path("/proc/mounts").read_text(encoding="utf-8").splitlines()
    except Exception:
        return False
    path_text = str(path)
    return any(len(line.split()) > 1 and line.split()[1] == path_text for line in mounts)


def maybe_mount_drive(enabled: bool, force_remount: bool, log_lines: list[str]) -> None:
    """Mount Google Drive when running inside Colab and the user requested it."""

    if not enabled:
        log_lines.append("Drive mount skipped by user.")
        return
    mount_point = Path("/content/drive")
    try:
        from google.colab import drive
    except Exception as exc:
        log_lines.append(f"Google Drive mount unavailable in this environment: {exc}")
        return

    if force_remount:
        log_lines.append("Force-remounting Google Drive before run.")
        # SAFETY: flush_and_unmount() only detaches the Drive FUSE - it never deletes
        # files. NEVER shutil.rmtree("/content/drive"): if the mount is still live
        # that would delete REAL files on Google Drive. Unmount first so the mount
        # point is empty, then remount.
        try:
            drive.flush_and_unmount()
            log_lines.append("Existing Google Drive mount flushed/unmounted.")
        except Exception as exc:
            log_lines.append(f"Drive flush/unmount skipped or failed: {exc}")
        try:
            drive.mount(str(mount_point), force_remount=True)
        except ValueError as exc:
            raise RuntimeError(
                f"Could not force-remount Google Drive ({exc}). Restart the Colab "
                "runtime (Runtime -> Disconnect and delete runtime) and run again with "
                "the force-remount checkbox OFF."
            )
        log_lines.append("Google Drive force-remounted.")
        return

    if Path("/content/drive/MyDrive").exists():
        log_lines.append("Google Drive already mounted.")
        return
    log_lines.append("Mounting Google Drive. Complete the auth prompt in the notebook output if it appears.")
    drive.mount(str(mount_point))


def describe_missing_input(input_path: Path, input_dir: Path) -> str:
    """Build a useful error message when an input path is not visible."""

    drive_root = Path("/content/drive/MyDrive")
    parent = input_path.parent
    search_dir = parent if parent.exists() else input_dir
    lines = [
        f"Input file not found: {input_path}",
        f"Google Drive mounted: {drive_root.exists()}",
        f"Requested parent exists: {parent.exists()}",
        "Colab paths are case-sensitive; check the exact file extension too.",
        "If the file is visible in Google Drive web UI but not here, force-remount Drive in Colab.",
        "Remount snippet: from google.colab import drive; drive.flush_and_unmount(); drive.mount('/content/drive', force_remount=True)",
    ]

    lines.append(f"Visible files in {search_dir}:")
    if search_dir.exists():
        visible_files = sorted(path for path in search_dir.iterdir() if path.is_file())
        if visible_files:
            lines.extend(f"- {path.name}" for path in visible_files[:50])
            if len(visible_files) > 50:
                lines.append(f"- ... {len(visible_files) - 50} more files")
        else:
            lines.append("- <no files>")
    else:
        lines.append("- <directory does not exist>")

    if input_dir.exists():
        similar_files = sorted(input_dir.rglob(f"{input_path.stem}.*"))
        if similar_files:
            lines.append("Files with the same name stem under project input:")
            lines.extend(f"- {path}" for path in similar_files[:20])
    return "\n".join(lines)


def read_scores(output_dir: Path):
    """Read aggregate and per-segment marketing score tables."""

    import pandas as pd

    scores_path = output_dir / "marketing_scores.csv"
    terms_path = output_dir / "decoded_terms.csv"
    report_path = output_dir / "report.json"
    missing = [str(path) for path in (scores_path, terms_path, report_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError("Decode finished without expected output files: " + ", ".join(missing))

    scores = pd.read_csv(scores_path)
    if scores.empty:
        raise RuntimeError(f"Marketing scores file is empty: {scores_path}")

    aggregate = scores[scores["time_index"].astype(str) == "aggregate"].copy()
    aggregate = aggregate.sort_values(["map_id", "mean_r"], ascending=[True, False])
    if aggregate.empty:
        raise RuntimeError(f"No aggregate rows found in {scores_path}")
    aggregate = aggregate.drop(columns=["score_0_100"], errors="ignore").rename(columns={"mean_r": "group_r"})

    segment_scores = scores[
        (scores["map_id"] == "tribe_predictions_fsaverage5")
        & (scores["time_index"].astype(str) != "aggregate")
    ].copy()
    if segment_scores.empty:
        return aggregate, pd.DataFrame()

    segment_scores["time_index_int"] = segment_scores["time_index"].astype(int)
    segment_scores = segment_scores.sort_values(
        ["time_index_int", "mean_r"], ascending=[True, False]
    )
    rows = []
    for time_index, group_df in segment_scores.groupby("time_index_int", sort=True):
        for rank, (_, row) in enumerate(group_df.head(3).iterrows(), start=1):
            rows.append(
                {
                    "segment": int(time_index),
                    "rank": rank,
                    "group": row["group"],
                    "group_r": float(row["mean_r"]),
                    "n_features": int(row["n_features"]),
                }
            )
    return aggregate, pd.DataFrame(rows)


def patch_exca_no_value_compat(log_lines: list[str] | None = None) -> None:
    """Restore the exca API path expected by neuralset 0.0.2."""

    try:
        from exca.steps import base, identity
    except Exception as exc:
        if log_lines is not None:
            log_lines.append(f"Could not inspect exca compatibility: {type(exc).__name__}: {exc}")
        return

    if not hasattr(base, "NoValue") and hasattr(identity, "NoValue"):
        base.NoValue = identity.NoValue
        if log_lines is not None:
            log_lines.append("Patched exca.steps.base.NoValue from exca.steps.identity.NoValue.")


def load_plot_segments(tribe_dir: Path, n_timesteps: int, log_lines: list[str]):
    """Load full TRIBE segment objects for the official demo visualization."""

    pkl_path = tribe_dir / "tribe_segments.pkl"
    if pkl_path.is_file():
        try:
            with pkl_path.open("rb") as handle:
                segments = list(pickle.load(handle))
            if segments:
                log_lines.append(f"Loaded full TRIBE segments for official plot: {pkl_path}")
                return segments[:n_timesteps], True
            log_lines.append(f"Full TRIBE segments file is empty: {pkl_path}")
        except Exception as exc:
            log_lines.append(f"Could not load full TRIBE segments from {pkl_path}: {type(exc).__name__}: {exc}")

    segments_path = tribe_dir / "tribe_segments.tsv"
    if segments_path.is_file():
        log_lines.append(
            "Only tribe_segments.tsv is available; video/audio/text rows need tribe_segments.pkl. "
            "Rerun TRIBE once with official stimuli plot enabled to create it."
        )
    else:
        log_lines.append(f"TRIBE segments not found: {segments_path}")
    return None, False


def make_tribe_plot(tribe_dir: Path, n_timesteps: int, try_show_stimuli: bool, log_lines: list[str]):
    """Create official TRIBE timestep visualization from cached predictions."""

    import numpy as np

    preds_path = tribe_dir / "tribe_predictions_fsaverage5.npy"
    if not preds_path.is_file():
        log_lines.append(f"Visualization skipped: predictions not found: {preds_path}")
        return None

    preds = np.load(preds_path).astype(np.float32)
    n_timesteps = max(1, min(int(n_timesteps), preds.shape[0]))

    try:
        patch_exca_no_value_compat(log_lines)
        from tribev2.plotting import PlotBrain

        plotter = PlotBrain(mesh="fsaverage5")
        segments, has_full_segments = load_plot_segments(tribe_dir, n_timesteps, log_lines)
        if try_show_stimuli and has_full_segments:
            try:
                return plotter.plot_timesteps(
                    preds[:n_timesteps],
                    segments=segments[:n_timesteps],
                    cmap="fire",
                    norm_percentile=99,
                    vmin=0.6,
                    alpha_cmap=(0, 0.2),
                    show_stimuli=True,
                )
            except Exception as exc:
                log_lines.append(
                    "Official TRIBE plot with stimuli failed; retrying brain-only plot: "
                    f"{type(exc).__name__}: {exc}"
                )
        elif try_show_stimuli:
            log_lines.append("Official stimuli plot skipped because full TRIBE segments are not available.")

        return plotter.plot_timesteps(
            preds[:n_timesteps],
            segments=None,
            cmap="fire",
            norm_percentile=99,
            vmin=0.6,
            alpha_cmap=(0, 0.2),
            show_stimuli=False,
        )
    except Exception as exc:
        log_lines.append(f"TRIBE plotter unavailable; using fallback heatmap: {type(exc).__name__}: {exc}")
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(10, 3.5))
        im = ax.imshow(preds[:n_timesteps], aspect="auto", interpolation="nearest", cmap="inferno")
        ax.set_title("TRIBE fsaverage5 predictions, fallback heatmap")
        ax.set_xlabel("fsaverage5 vertex")
        ax.set_ylabel("segment")
        fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
        return fig



def persist_run_log(log_path: Path | None, log_lines: list[str]) -> None:
    """Persist the latest UI log to Drive/local disk for debugging."""

    if log_path is None:
        return
    try:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("\n".join(log_lines), encoding="utf-8")
    except Exception:
        pass


def describe_output_dir(output_dir: Path) -> str:
    """Return a compact listing of output files for debug logs."""

    if not output_dir.exists():
        return f"Output dir does not exist: {output_dir}"
    rows = []
    for path in sorted(output_dir.glob("*")):
        if path.is_file():
            rows.append(f"{path.name}: {path.stat().st_size} bytes")
        else:
            rows.append(f"{path.name}/")
    return "\n".join(rows) if rows else f"Output dir is empty: {output_dir}"


def cached_transcription_matches(
    tribe_dir: Path,
    backend: str,
    target_language: str,
    gigaam_model: str,
    openrouter_model: str,
) -> bool:
    """Check whether cached TRIBE outputs were built with the selected transcript backend."""

    metadata_path = tribe_dir / "tribe_transcription_backend.json"
    if not metadata_path.is_file():
        return backend == "tribe"
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception:
        return False

    if metadata.get("backend") != backend:
        return False
    if backend != "hybrid":
        return True
    return (
        metadata.get("target_language") == target_language
        and metadata.get("gigaam_model") == gigaam_model
        and metadata.get("openrouter_model") == openrouter_model
    )


def analyze_video(
    video_path: str,
    hf_token: str,
    openrouter_api_key: str,
    transcript_backend: str,
    transcript_target_language: str,
    gigaam_model: str,
    openrouter_model: str,
    root_dir: str,
    install_requirements: bool,
    mount_drive: bool,
    force_remount_drive: bool,
    force_rerun_tribe: bool,
    use_persistent_worker: bool,
    n_timesteps: int,
    show_stimuli: bool,
    progress=gr.Progress(),
):
    """Gradio generator that performs setup, TRIBE, decode and visualization."""

    import pandas as pd

    log_lines: list[str] = []
    empty_df = pd.DataFrame()
    log_path: Path | None = None

    def snapshot(status: str, fig=None, aggregate=None, segments=None, paths_text: str = "", report_file=None):
        persist_run_log(log_path, log_lines)
        return (
            status,
            "\n".join(log_lines[-320:]),
            paths_text,
            fig,
            aggregate if aggregate is not None else empty_df,
            segments if segments is not None else empty_df,
            report_file,
        )

    try:
        os.environ.setdefault("PYTHONUTF8", "1")
        os.environ.setdefault("PYTHONUNBUFFERED", "1")
        write_project_files()
        log_lines.append(f"Notebook build: {DEBUG_BUILD_ID}")
        log_lines.append(f"Project files written to {PROJECT_DIR}")
        yield snapshot("Project files ready")

        if mount_drive:
            maybe_mount_drive(True, force_remount_drive, log_lines)
            yield snapshot("Drive checked")

        token = (hf_token or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            os.environ["HUGGING_FACE_HUB_TOKEN"] = token
            log_lines.append("HF token loaded from UI field.")
        else:
            log_lines.append("HF token field is empty. Cached/public models may still work.")

        backend = (transcript_backend or "hybrid").strip().lower()
        if backend not in {"hybrid", "tribe"}:
            raise ValueError(f"Unknown transcription backend: {transcript_backend}")
        target_language = (transcript_target_language or "en").strip() or "en"
        selected_gigaam_model = (gigaam_model or "v3_e2e_rnnt").strip() or "v3_e2e_rnnt"        
        selected_openrouter_model = (
            (openrouter_model or "google/gemini-3.5-flash").strip()
            or "google/gemini-3.5-flash"
        )
        openrouter_token = (openrouter_api_key or "").strip()
        if openrouter_token:
            os.environ["OPENROUTER_API_KEY"] = openrouter_token
            log_lines.append("OpenRouter API key loaded from UI field.")
        elif backend == "hybrid" and not os.environ.get("OPENROUTER_API_KEY"):
            raise RuntimeError(
                "OpenRouter API key is required for hybrid transcription. "
                "Paste it in the UI field or set OPENROUTER_API_KEY in the environment."
            )
        log_lines.append(
            "Transcription backend: "
            f"{backend}; target_language={target_language}; "
            f"GigaAM={selected_gigaam_model}; correction_model={selected_openrouter_model}"
        )

        env = os.environ.copy()
        env["PYTHONUTF8"] = "1"
        env["PYTHONUNBUFFERED"] = "1"

        root = Path(root_dir).expanduser()
        log_path = root / "logs" / "gradio_last_run.log"
        log_lines.append(f"Debug log will be saved to {log_path}")
        append_resource_snapshot(log_lines, "initial notebook state")
        yield snapshot("Diagnostics ready")

        if install_requirements:
            log_lines.append(
                "Installing/updating Python requirements. This can take several minutes on a fresh Colab runtime. "
                "TRIBE packages are installed in a second --no-deps step to avoid their invalid NumPy pins."
            )
            install_cmd = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")]
            for _ in run_command_stream(install_cmd, log_lines, env=env):
                yield snapshot("Installing requirements")
            tribe_install_cmd = [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-deps",
                "-r",
                str(PROJECT_DIR / "requirements-tribe-nodeps.txt"),
            ]
            for _ in run_command_stream_with_retries(tribe_install_cmd, log_lines, env=env):
                yield snapshot("Installing TRIBE packages")
            hybrid_install_cmd = [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-deps",
                "--force-reinstall",
                "-r",
                str(PROJECT_DIR / "requirements-hybrid-nodeps.txt"),
            ]
            for _ in run_command_stream_with_retries(hybrid_install_cmd, log_lines, env=env):
                yield snapshot("Installing hybrid transcription package")
            hybrid_check_cmd = [
                sys.executable,
                "-c",
                "import onnxruntime, gigaam; print('hybrid import check ok', onnxruntime.__version__)",
            ]
            for _ in run_command_stream(hybrid_check_cmd, log_lines, env=env):
                yield snapshot("Checking hybrid transcription imports")
        else:
            log_lines.append("Requirements install skipped by user.")

        input_dir = root / "input"
        output_root = root / "outputs" / "surface_decoder"
        tribe_output_root = root / "outputs" / "tribe_fsaverage5"
        tribe_cache_dir = root / "cache" / "tribev2"
        references_dir = root / "cache" / "surface_references" / "marketing_v2"
        neurosynth_cache_dir = root / "cache" / "neurosynth_precomputed"
        for path in (input_dir, output_root, tribe_output_root, tribe_cache_dir, references_dir, neurosynth_cache_dir):
            path.mkdir(parents=True, exist_ok=True)

        requested_path = Path(video_path).expanduser()
        batch_mode = requested_path.is_dir()
        if batch_mode:
            input_files = sorted(
                path
                for path in requested_path.iterdir()
                if path.is_file() and path.suffix.lower() in BATCH_VIDEO_EXTENSIONS
            )
            _mp4_stems = {p.stem for p in input_files if p.suffix.lower() == ".mp4"}
            input_files = [
                p
                for p in input_files
                if not (p.suffix.lower() == ".avi" and p.stem in _mp4_stems)
            ]
            if not input_files:
                raise FileNotFoundError(
                    f"No .mp4/.avi files found in input directory: {requested_path}"
                )
            paths_text = (
                f"Batch input folder: {requested_path}\n"
                f"MP4 videos found: {len(input_files)}\n"
                f"Batch output root: {output_root}"
            )
            log_lines.append(f"Batch input folder: {requested_path}")
            log_lines.append(f"Found {len(input_files)} .mp4/.avi file(s) for sequential processing.")
        else:
            if not requested_path.is_file():
                raise FileNotFoundError(describe_missing_input(requested_path, input_dir))
            if requested_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                raise ValueError(f"Unsupported input extension: {requested_path.suffix}")
            input_files = [requested_path]
            paths_text = f"Input: {requested_path}"
        yield snapshot("Input ready", paths_text=paths_text)

        reference_maps = references_dir / "reference_maps.npy"
        reference_metadata = references_dir / "reference_metadata.json"
        if not reference_maps.is_file() or not reference_metadata.is_file():
            log_lines.append("Reference maps not found. Building lightweight Neurosynth references...")
            build_cmd = [
                sys.executable,
                str(PROJECT_DIR / "marketing_surface_decoder.py"),
                "build-references",
                "--references-dir",
                str(references_dir),
                "--reference-source",
                "neurosynth_precomputed",
                "--neurosynth-cache-dir",
                str(neurosynth_cache_dir),
                "--n-cores",
                "1",
            ]
            for _ in run_command_stream(build_cmd, log_lines, env=env):
                yield snapshot("Building reference maps", paths_text=paths_text)
        else:
            log_lines.append(f"Using cached reference maps: {references_dir}")
            yield snapshot("Reference maps cached", paths_text=paths_text)

        processed_reports: list[tuple[Path, Path]] = []
        failed_reports: list[dict[str, str]] = []
        tribe_worker_session = None
        tribe_worker_restarts = 0
        final_fig = None
        final_aggregate_df = empty_df
        final_segment_top_df = empty_df
        final_paths_text = paths_text
        final_report_file: str | None = None
        total_inputs = len(input_files)
        try:
            progress(0.0, desc=f"0/{total_inputs} videos")
        except Exception:
            pass

        for file_index, input_path in enumerate(input_files, start=1):
            progress_prefix = f"[{file_index}/{total_inputs}] " if batch_mode else ""
            current_stage = "preparing paths"
            converted_tmp = None
            try:
                if input_path.suffix.lower() == ".avi":
                    current_stage = "converting avi to mp4"
                    conversions_dir = PROJECT_DIR / "tmp_conversions"
                    conversions_dir.mkdir(parents=True, exist_ok=True)
                    converted_tmp = conversions_dir / f"{input_path.stem}.mp4"
                    log_lines.append(f"{progress_prefix}Converting AVI to MP4 (temporary): {input_path.name}")
                    convert_cmd = [
                        "ffmpeg",
                        "-y",
                        "-i",
                        str(input_path),
                        "-c:v",
                        "libx264",
                        "-preset",
                        "veryfast",
                        "-pix_fmt",
                        "yuv420p",
                        "-c:a",
                        "aac",
                        str(converted_tmp),
                    ]
                    for _ in run_command_stream(convert_cmd, log_lines, env=env):
                        yield snapshot(
                            f"{progress_prefix}Converting AVI to MP4",
                            paths_text=f"{progress_prefix}Converting {input_path.name} to MP4",
                        )
                    if not (converted_tmp.is_file() and converted_tmp.stat().st_size > 0):
                        raise FileNotFoundError(
                            f"AVI to MP4 conversion produced no output: {converted_tmp}"
                        )
                    input_path = converted_tmp
                current_stage = "preparing paths"
                tribe_dir = output_dir_for_input(input_path, input_dir, tribe_output_root)
                surface_dir = output_root.joinpath(*tribe_dir.relative_to(tribe_output_root).parts)
                report_html = surface_dir / "marketing_report.html"
                report_zip = surface_dir / "marketing_report_bundle.zip"
                tribe_dir.mkdir(parents=True, exist_ok=True)
                surface_dir.mkdir(parents=True, exist_ok=True)
                paths_text = (
                    f"{progress_prefix}Input: {input_path}\n"
                    f"TRIBE output: {tribe_dir}\n"
                    f"Decode output: {surface_dir}\n"
                    f"Scores: {surface_dir / 'marketing_scores.csv'}\n"
                    f"HTML report: {report_html}\n"
                    f"Download bundle: {report_zip}"
                )
                final_paths_text = paths_text
                yield snapshot(f"{progress_prefix}Paths ready", paths_text=paths_text)

                current_stage = "checking TRIBE cache"
                prediction_file = tribe_dir / "tribe_predictions_fsaverage5.npy"
                activity_file = tribe_dir / "tribe_activity_fsaverage5.npy"
                segments_pickle_file = tribe_dir / "tribe_segments.pkl"
                has_tribe_cache = prediction_file.is_file() and activity_file.is_file()
                cache_backend_ok = cached_transcription_matches(
                    tribe_dir=tribe_dir,
                    backend=backend,
                    target_language=target_language,
                    gigaam_model=selected_gigaam_model,
                    openrouter_model=selected_openrouter_model,
                )
                needs_full_segments = bool(show_stimuli)
                can_reuse_tribe = (
                    has_tribe_cache
                    and cache_backend_ok
                    and not force_rerun_tribe
                    and (not needs_full_segments or segments_pickle_file.is_file())
                )
                if can_reuse_tribe:
                    log_lines.append(f"Using cached TRIBE outputs: {tribe_dir}")
                    yield snapshot(f"{progress_prefix}Using cached TRIBE outputs", paths_text=paths_text)
                else:
                    if has_tribe_cache and not cache_backend_ok:
                        log_lines.append(
                            "Existing TRIBE cache was built with different transcription settings. "
                            "Rerunning TRIBE to avoid mixing WhisperX/old text events with hybrid outputs."
                        )
                    if has_tribe_cache and not force_rerun_tribe and needs_full_segments and not segments_pickle_file.is_file():
                        log_lines.append(
                            "Cached .npy outputs exist, but full tribe_segments.pkl is missing. "
                            "Rerunning TRIBE once so the official video/audio/text visualization can be built. "
                            "Disable the stimuli checkbox to reuse old cache without rerun."
                        )
                    if use_persistent_worker:
                        tribe_job = {
                            "input": str(input_path),
                            "output_dir": str(tribe_dir),
                            "gigaam_download_root": str(tribe_cache_dir),
                            "backend": backend,
                            "target_language": target_language,
                            "gigaam_model": selected_gigaam_model,
                            "openrouter_model": selected_openrouter_model,
                            "aggregation": "mean",
                        }
                        worker_alive = (
                            tribe_worker_session is not None
                            and tribe_worker_session.process is not None
                            and tribe_worker_session.process.poll() is None
                        )
                        if not worker_alive:
                            if tribe_worker_session is not None:
                                tribe_worker_restarts += 1
                                if tribe_worker_restarts > TRIBE_WORKER_MAX_RESTARTS:
                                    raise RuntimeError(
                                        f"TRIBE worker exceeded {TRIBE_WORKER_MAX_RESTARTS} restarts; failing this video."
                                    )
                                log_lines.append(
                                    f"{progress_prefix}Restarting TRIBE worker (restart {tribe_worker_restarts})..."
                                )
                            current_stage = "loading TRIBE worker"
                            worker_cmd = [
                                sys.executable,
                                str(PROJECT_DIR / "tribe_worker.py"),
                                "--checkpoint",
                                "facebook/tribev2",
                                "--cache-dir",
                                str(tribe_cache_dir),
                                "--device",
                                "cuda",
                            ]
                            tribe_worker_session = TribeWorkerSession(worker_cmd, log_lines, env=env)
                            log_lines.append(f"{progress_prefix}Starting persistent TRIBE worker (loads model once)...")
                            for _ in tribe_worker_session.start():
                                yield snapshot(f"{progress_prefix}Loading TRIBE model", paths_text=paths_text)
                        current_stage = "running TRIBE v2 (worker)"
                        log_lines.append(f"{progress_prefix}Running TRIBE v2 (persistent worker)...")
                        for _ in tribe_worker_session.run_job(tribe_job):
                            yield snapshot(f"{progress_prefix}Running TRIBE v2", paths_text=paths_text)
                        worker_result = tribe_worker_session.last_result or {}
                        if worker_result.get("status") != "ok":
                            raise RuntimeError(
                                f"TRIBE worker job failed: {worker_result.get('error', worker_result)}"
                            )
                    else:
                        tribe_cmd = [
                            sys.executable,
                            str(PROJECT_DIR / "tribe_nimare_interpreter.py"),
                            str(input_path),
                            "--output-dir",
                            str(tribe_dir),
                            "--cache-dir",
                            str(tribe_cache_dir),
                            "--device",
                            "cuda",
                            "--tribe-only",
                            "--transcript-backend",
                            backend,
                            "--transcript-target-language",
                            target_language,
                            "--gigaam-model",
                            selected_gigaam_model,
                            "--gigaam-download-root",
                            str(tribe_cache_dir),
                            "--openrouter-model",
                            selected_openrouter_model,
                        ]
                        current_stage = "running TRIBE v2"
                        log_lines.append(f"{progress_prefix}Running TRIBE v2...")
                        for _ in run_command_stream(tribe_cmd, log_lines, env=env):
                            yield snapshot(f"{progress_prefix}Running TRIBE v2", paths_text=paths_text)

                current_stage = "collecting TRIBE outputs"
                decode_inputs = []
                if prediction_file.is_file():
                    decode_inputs.append(prediction_file)
                if activity_file.is_file():
                    decode_inputs.append(activity_file)
                if not decode_inputs:
                    raise FileNotFoundError(f"No TRIBE .npy outputs found in {tribe_dir}")

                decode_cmd = [
                    sys.executable,
                    str(PROJECT_DIR / "marketing_surface_decoder.py"),
                    "decode",
                    *[str(path) for path in decode_inputs],
                    "--references-dir",
                    str(references_dir),
                    "--output-dir",
                    str(surface_dir),
                    "--hemisphere-order",
                    "left_then_right",
                ]
                current_stage = "running surface decoder"
                log_lines.append(f"{progress_prefix}Running surface decoder...")
                for _ in run_command_stream(decode_cmd, log_lines, env=env):
                    yield snapshot(f"{progress_prefix}Decoding", paths_text=paths_text)

                log_lines.append("Decode output files:")
                log_lines.append(describe_output_dir(surface_dir))
                current_stage = "reading decoder scores"
                aggregate_df, segment_top_df = read_scores(surface_dir)
                log_lines.append(f"Loaded aggregate rows: {len(aggregate_df)}; segment top rows: {len(segment_top_df)}")

                current_stage = "building report"
                report_cmd = [
                    sys.executable,
                    str(PROJECT_DIR / "marketing_report.py"),
                    "--tribe-dir",
                    str(tribe_dir),
                    "--surface-dir",
                    str(surface_dir),
                    "--output-html",
                    str(report_html),
                    "--output-zip",
                    str(report_zip),
                    "--input-media",
                    str(input_path),
                    "--title",
                    f"TRIBE v2 marketing report: {input_path.name}",
                ]
                log_lines.append(f"{progress_prefix}Building downloadable HTML + Excel report bundle with all TRIBE timesteps...")
                for _ in run_command_stream(report_cmd, log_lines, env=env):
                    yield snapshot(f"{progress_prefix}Building report", paths_text=paths_text)
                current_stage = "validating report outputs"
                if not report_html.is_file():
                    raise FileNotFoundError(f"HTML report was not created: {report_html}")
                if not report_zip.is_file():
                    raise FileNotFoundError(f"Report ZIP was not created: {report_zip}")

                processed_reports.append((input_path, report_zip))
                final_report_file = str(report_zip)
                final_aggregate_df = aggregate_df
                final_segment_top_df = segment_top_df
                if file_index == total_inputs:
                    final_fig = make_tribe_plot(tribe_dir, n_timesteps, show_stimuli, log_lines)
                if batch_mode:
                    yield snapshot(
                        f"{progress_prefix}Video done",
                        fig=final_fig,
                        aggregate=final_aggregate_df,
                        segments=final_segment_top_df,
                        paths_text=paths_text,
                        report_file=final_report_file,
                    )

            except Exception as exc:
                if not batch_mode:
                    raise
                failure_traceback = traceback.format_exc()
                failed_reports.append(
                    {
                        "input": str(input_path),
                        "name": input_path.name,
                        "stage": current_stage,
                        "error_type": type(exc).__name__,
                        "error": str(exc),
                        "traceback": failure_traceback,
                    }
                )
                log_lines.append(f"{progress_prefix}FAILED: {input_path}")
                log_lines.append(f"{progress_prefix}Failed stage: {current_stage}")
                log_lines.append(failure_traceback)
                yield snapshot(
                    f"{progress_prefix}Video failed; continuing",
                    fig=final_fig,
                    aggregate=final_aggregate_df,
                    segments=final_segment_top_df,
                    paths_text=paths_text,
                    report_file=final_report_file,
                )
                continue
            finally:
                try:
                    progress(file_index / total_inputs, desc=f"{file_index}/{total_inputs} videos processed")
                except Exception:
                    pass
                if converted_tmp is not None:
                    try:
                        converted_tmp.unlink()
                        log_lines.append(f"{progress_prefix}Removed temporary MP4: {converted_tmp.name}")
                    except FileNotFoundError:
                        pass
                    except Exception as cleanup_error:
                        log_lines.append(f"{progress_prefix}Could not remove temporary MP4 {converted_tmp}: {cleanup_error}")
        if tribe_worker_session is not None:
            tribe_worker_session.shutdown()
        if batch_mode:
            if not processed_reports:
                failure_summary = "\n".join(f"- {item['name']}: {item['stage']} -> {item['error']}" for item in failed_reports)
                raise RuntimeError(f"Batch failed: 0/{total_inputs} videos processed.\n{failure_summary}")
            batch_zip_name = f"{requested_path.name or 'batch'}_reports.zip"
            batch_zip = output_root / batch_zip_name
            batch_zip.parent.mkdir(parents=True, exist_ok=True)
            batch_failures_path = output_root / f"{requested_path.name or 'batch'}_failures.json"
            batch_failures_manifest = {
                "input_folder": str(requested_path),
                "total_videos": total_inputs,
                "processed_count": len(processed_reports),
                "failed_count": len(failed_reports),
                "failures": failed_reports,
            }
            batch_failures_path.write_text(
                json.dumps(batch_failures_manifest, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            with zipfile.ZipFile(batch_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
                for input_path, report_zip in processed_reports:
                    archive.write(report_zip, arcname=input_path.with_suffix(".zip").name)
                archive.write(batch_failures_path, arcname="batch_failures.json")
            final_report_file = str(batch_zip)
            included_reports = "\n".join(
                f"- {input_path.name} -> {input_path.with_suffix('.zip').name}"
                for input_path, _ in processed_reports
            )
            final_paths_text = (
                f"Batch input folder: {requested_path}\n"
                f"Processed videos: {len(processed_reports)}\n"
                f"Failed videos: {len(failed_reports)}\n"
                f"Batch ZIP: {batch_zip}\n"
                f"Failures manifest: {batch_failures_path}\n"
                f"Included report ZIPs:\n{included_reports}"
            )
            log_lines.append(f"Batch report ZIP created: {batch_zip}")
            if failed_reports:
                log_lines.append(f"Batch completed with {len(failed_reports)} failed video(s). See {batch_failures_path}")
            yield snapshot(
                "Batch done" if not failed_reports else "Batch done with failures",
                fig=final_fig,
                aggregate=final_aggregate_df,
                segments=final_segment_top_df,
                paths_text=final_paths_text,
                report_file=final_report_file,
            )
        else:
            yield snapshot(
                "Done",
                fig=final_fig,
                aggregate=final_aggregate_df,
                segments=final_segment_top_df,
                paths_text=final_paths_text,
                report_file=final_report_file,
            )
    except Exception:
        log_lines.append(traceback.format_exc())
        persist_run_log(log_path, log_lines)
        yield snapshot("Failed")


with gr.Blocks(title="TRIBE v2 Surface Decoder") as demo:
    gr.Markdown("# TRIBE v2 + Surface Marketing Decoder")
    gr.Markdown("Choose one input file, or a folder with `.mp4`/`.avi` videos for sequential batch processing (AVI is auto-converted to MP4 in a temporary file that is deleted after processing). Nothing is processed until you click **Run analysis**. Default transcription backend is **hybrid**: GigaAM builds word timings, then an OpenRouter multimodal model corrects/translates the words before TRIBE inference. The old TRIBE/WhisperX path is available only if you select `tribe`. The downloadable ZIP contains HTML + Excel reports with all TRIBE timesteps, not only the preview limit. In folder mode, the final ZIP contains one report ZIP per video.")

    with gr.Row():
        video_path_input = gr.Textbox(label="Input file or folder path", value=DEFAULT_INPUT_PATH, lines=1)
        root_dir_input = gr.Textbox(label="Project root", value=DEFAULT_ROOT_DIR, lines=1)
    hf_token_input = gr.Textbox(label="Hugging Face token (optional, not saved)", type="password", lines=1)
    openrouter_api_key_input = gr.Textbox(
        label="OpenRouter API key (optional, not saved)",
        type="password",
        lines=1,
        placeholder="Required for hybrid unless OPENROUTER_API_KEY is already set",
    )

    with gr.Row():
        transcript_backend_input = gr.Dropdown(
            label="Transcription backend",
            choices=["hybrid", "tribe"],
            value="hybrid",
        )
        transcript_target_language_input = gr.Dropdown(
            label="Corrected transcript target language",
            choices=["en", "ru"],
            value="en",
            allow_custom_value=True,
        )
    with gr.Row():
        gigaam_model_input = gr.Dropdown(
            label="GigaAM timing model",
            choices=["v3_e2e_rnnt", "v3_e2e_ctc", "v2_rnnt", "rnnt", "v1_rnnt", "ctc", "v2_ctc"],
            value="v3_e2e_rnnt",
            allow_custom_value=True,
        )        
        openrouter_model_input = gr.Textbox(
            label="OpenRouter correction/translation model",
            value="google/gemini-3.5-flash",
            lines=1,
        )

    with gr.Row():
        install_requirements_input = gr.Checkbox(label="Install/update requirements before run", value=True)
        mount_drive_input = gr.Checkbox(label="Mount Google Drive if needed", value=True)
        force_remount_drive_input = gr.Checkbox(label="Force Google Drive remount before run", value=True)
        force_rerun_input = gr.Checkbox(label="Recompute TRIBE even if cache exists", value=False)
        persistent_worker_input = gr.Checkbox(label="Persistent TRIBE worker (load model once per run)", value=True)
    with gr.Row():
        n_timesteps_input = gr.Slider(label="TRIBE timesteps to plot", minimum=1, maximum=15, value=15, step=1)
        show_stimuli_input = gr.Checkbox(label="Use official TRIBE plot with video/audio/text stimuli", value=True)

    run_button = gr.Button("Run analysis", variant="primary")

    status_output = gr.Textbox(label="Status", lines=1)
    log_output = gr.Textbox(label="Progress log", lines=22)
    paths_output = gr.Textbox(label="Output paths", lines=4)
    plot_output = gr.Plot(label="TRIBE timestep visualization")
    aggregate_output = gr.Dataframe(label="Aggregate marketing scores")
    segment_output = gr.Dataframe(label="Top-3 categories per segment")
    report_file_output = gr.File(label="Download report ZIP / batch ZIP")

    run_button.click(
        fn=analyze_video,
        inputs=[
            video_path_input,
            hf_token_input,
            openrouter_api_key_input,
            transcript_backend_input,
            transcript_target_language_input,
            gigaam_model_input,
            openrouter_model_input,
            root_dir_input,
            install_requirements_input,
            mount_drive_input,
            force_remount_drive_input,
            force_rerun_input,
            persistent_worker_input,
            n_timesteps_input,
            show_stimuli_input,
        ],
        outputs=[status_output, log_output, paths_output, plot_output, aggregate_output, segment_output, report_file_output],
    )

server_port = find_open_port()
local_url = f"http://127.0.0.1:{server_port}"

prelaunch_url_text = (
    f"local_url={local_url}\n"
    "share_url=creating\n"
)
print("Starting Gradio launch. Waiting for gradio.live share_url...", flush=True)
print(prelaunch_url_text, flush=True)
save_gradio_urls(prelaunch_url_text)

queued_demo = demo.queue()
print(f"Queue initialized. Calling launch(...) on port {server_port}", flush=True)
launch_result = queued_demo.launch(
    share=True,
    debug=False,
    prevent_thread_lock=True,
    inline=False,
    quiet=False,
    show_error=True,
    server_name="0.0.0.0",
    server_port=server_port,
)

try:
    _, returned_local_url, share_url = launch_result
    local_url = returned_local_url or local_url
except Exception:
    local_url = getattr(launch_result, "local_url", None) or local_url
    share_url = getattr(launch_result, "share_url", None)

url_text = f"local_url={local_url}\nshare_url={share_url}\n"
print("Gradio is running.", flush=True)
print(url_text, flush=True)
print("Open share_url in a browser. If notebook output disappears, check gradio_url.txt.", flush=True)
save_gradio_urls(url_text)

_GRADIO_DEMO = demo
_GRADIO_LAUNCH_RESULT = launch_result
print("The cell can finish now; use the printed share_url to open the UI.", flush=True)


NumPy compatible with TRIBE plotting: 2.0.2
Importing Gradio...
Gradio imported: 5.50.0
Starting Gradio launch. Waiting for gradio.live share_url...
local_url=http://127.0.0.1:7860
share_url=creating

Saved Gradio URL to: /content/drive/MyDrive/neuromedia/logs/gradio_url.txt
Queue initialized. Calling launch(...) on port 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a88f2627f31e6cd702.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Gradio is running.
local_url=http://localhost:7860/
share_url=https://a88f2627f31e6cd702.gradio.live

Open share_url in a browser. If notebook output disappears, check gradio_url.txt.
Saved Gradio URL to: /content/drive/MyDrive/neuromedia/logs/gradio_url.txt
The cell can finish now; use the printed share_url to 